In [ ]:
import pygmt
import xarray as xr
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np

gebco=xr.open_dataset("GEBCO_2025.nc")
sediment_thickness_grid = xr.open_dataset("GlobSed-v3.nc")

lon=np.arange(-180,180.1,0.1)
lat=np.arange(-90,90.1,0.1)

gebco_interp = gebco.interp(lon=lon,lat=lat, method="linear", kwargs={"fill_value": "extrapolate"})
sediment_interp = sediment_thickness_grid.interp(lon=lon,lat=lat)
sediment_interp = sediment_interp.where(sediment_interp['z'] >= 1e-4 , np.nan)

basement_depth=gebco_interp['elevation']-sediment_interp['z']



In [ ]:
import scipy.special as special
Rou_sg=2.647
Fai_0=0.66
Lambda=1.33
Rou_w=1.03

z=sediment_interp['z']/1000
Rou_intergation=2.647 * z- 1.4229138 * np.log(z) + 1.4229138 * special.expi(-0.75187969924812 * z)
Rou_0 = 2.647 * 0.0000001- 1.4229138 * np.log(0.0000001) + 1.4229138 * special.expi(-0.75187969924812 * 0.0000001)
average_density=(Rou_intergation-Rou_0)/z


In [5]:
Rou_a=3.2
iso_correction=z*(average_density - Rou_a)/(Rou_w-Rou_a) 
iso_corrected_bathymetry=(gebco_interp['elevation']-iso_correction*1000)

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

crust_thickness=pd.read_csv('TotalThickness_ECM1.txt',delim_whitespace=True)

/var/folders/cr/63588mrd1yv77xydhhp7lgkw0000gp/T/ipykernel_43816/1180774661.py:5: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  crust_thickness=pd.read_csv('/Users/yzha0335/Documents/Data/TotalThickness_ECM1.txt',delim_whitespace=True)


In [7]:

crust_thickness_xr=xr.DataArray(
    data=crust_thickness['TotalThickness'].values.reshape(180,360),
    dims=['lat','lon'],
    coords={'lat':crust_thickness['Lat'].unique(),'lon':crust_thickness['Lon'].unique()}
)

In [8]:
crust_interp=crust_thickness_xr.interp(lon=lon, lat=lat, method="linear", kwargs={"fill_value": "extrapolate"})

crust_interp

In [9]:
crustal_correction = (Rou_a-2.86)/(Rou_a - Rou_w) * (crust_interp-7.1)
corrected_bathymetry=(iso_corrected_bathymetry - crustal_correction*1000)

In [ ]:
Base_from_age= xr.open_dataarray('depth2base_richards_0.0.nc')

In [15]:
Base_from_age_interp = Base_from_age.interp(lon=lon,lat=lat)
Residual_anomaly = corrected_bathymetry-Base_from_age_interp

In [ ]:
Residual_anomaly.name = "z"
Residual_anomaly.to_netcdf("RT_ECM1_Richards.nc")